# Translate `DeepPavlov/hwu_intent_classification` to French and Spanish

Translates the English HWU64 voice-assistant intent-classification dataset (alarms,
calendar, IoT, music, general commands, ...) into French and Spanish using two
locally-hosted OpenAI-compatible LLM servers (e.g. vLLM/SGLang), and saves the result
as one HF dataset with a **subset (config) per language**. Each subset has 4 columns:
`text` (original English), `label`, and one translation column per model
(`text_gemma`, `text_qwen`) so the two models' outputs can be compared directly.

HWU64 utterances are lowercase, mostly-grammatical voice-assistant commands, sometimes
with disfluencies/duplicated words ("see see for me the alarms...") and references to
device/brand/wake-word names (Alexa-style "olly", "wemo", "domino's", "snapchat", ...)
that must stay untranslated. The prompt and few-shot examples below are tuned for that.

- `gemma` — `google/gemma-4-31B-it` on `http://localhost:8088/v1`
- `qwen`  — `Qwen/Qwen3.6-27B-FP8` on `http://localhost:8000/v1`

**Setup.** This repo's `uv` environment already has `datasets`; it does not have
`openai`. Launch this notebook with the extra dependency pulled in on the fly, without
touching `pyproject.toml`:

```bash
uv run --with openai --with ipykernel jupyter lab
```

Translation is checkpointed to `translations/hwu64/*.jsonl` (one line per example),
so the notebook is safe to interrupt and re-run — already-translated rows are skipped.


In [1]:
import json
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from datasets import Dataset, DatasetDict, load_dataset
from openai import OpenAI
from tqdm.auto import tqdm


## Config

Both models translate into both languages. Adjust ports/model names/languages here.

In [2]:
MODELS = {
    "gemma": {"base_url": "http://localhost:8088/v1", "model": "google/gemma-4-31B-it"},
    "qwen": {
        "base_url": "http://localhost:8000/v1",
        "model": "Qwen/Qwen3.6-27B-FP8",
        # Qwen3 is a hybrid-thinking model: without this it emits its chain-of-thought
        # as the actual response content instead of a final answer.
        "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
    },
}

LANGUAGES = {
    "fr": "French",
    "es": "Spanish",
}

OUT_DIR = Path("translations/hwu64")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 64
TEMPERATURE = 0.0


In [3]:
clients = {name: OpenAI(base_url=cfg["base_url"], api_key="EMPTY") for name, cfg in MODELS.items()}

for name, client in clients.items():
    available = [m.id for m in client.models.list().data]
    print(f"{name} ({MODELS[name]['base_url']}): serving {available}")
    assert MODELS[name]["model"] in available, (
        f"{MODELS[name]['model']} not found on {name} server; available: {available}"
    )


APIConnectionError: Connection error.

## Load the dataset

In [4]:
raw = load_dataset("DeepPavlov/hwu_intent_classification")
raw


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 8954
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 1076
    })
})

## Translation

The prompt asks for a natural, native-sounding translation that preserves the register
of the source: HWU64 utterances are lowercase, voice-assistant commands (alarms,
calendar, IoT, music, lists, general chit-chat with the assistant), occasionally with
disfluencies/duplicated words, and often referencing device/brand/wake-word names that
must stay untranslated (e.g. "olly", "wemo", "domino's", "snapchat"). Translations
should read like the same kind of spoken command in the target language -- lowercase,
not over-formalized. A few hand-written examples are included as few-shot demonstrations,
per target language, to anchor that register.


In [5]:
# Few-shot examples per language: (english, translation). Kept lowercase to match the
# register of HWU64 utterances (voice-assistant commands). Time expressions keep their
# raw source format and drop the "am"/"pm" marker (not standard in French/Spanish),
# matching the same convention used for the ATIS notebook.
EXAMPLES = {
    "fr": [
        (
            "see see for me the alarms that you have set tomorrow morning",
            "montre montre-moi les alarmes que tu as reglees pour demain matin",
        ),
        (
            "what is the wake up time for my alarm i have set for the flight this weekend",
            "quelle est l'heure de reveil pour mon alarme que j'ai reglee pour le vol ce week-end",
        ),
        (
            "do i have an alarm set for morning flight",
            "est-ce que j'ai une alarme reglee pour le vol du matin",
        ),
        (
            "how many alarms do i have set for morning hours between six and nine am",
            "combien d'alarmes j'ai reglees pour les heures du matin entre six et neuf",
        ),
    ],
    "es": [
        (
            "see see for me the alarms that you have set tomorrow morning",
            "muestrame muestrame las alarmas que has puesto para manana por la manana",
        ),
        (
            "what is the wake up time for my alarm i have set for the flight this weekend",
            "cual es la hora de despertar para mi alarma que he puesto para el vuelo este fin de semana",
        ),
        (
            "do i have an alarm set for morning flight",
            "tengo una alarma puesta para el vuelo de la manana",
        ),
        (
            "how many alarms do i have set for morning hours between six and nine am",
            "cuantas alarmas tengo puestas para las horas de la manana entre las seis y las nueve",
        ),
    ],
}


In [6]:
SYSTEM_PROMPT = (
    "You are a professional translator localizing voice-assistant commands (alarms, "
    "calendar, IoT/smart-home, music, lists, general chit-chat with the assistant; "
    "lowercase, casual speech-command style). Translate the user's message from English "
    "into {lang_name}. Keep the same meaning, tone, and register -- including the "
    "lowercase style and any disfluencies or duplicated words -- but produce something "
    "that reads naturally to a native {lang_name} speaker. "
    "Keep device names, brand names, app names, and wake words UNTRANSLATED exactly as "
    "written (e.g. 'olly', 'wemo', 'alexa', \"domino's\", 'snapchat', 'spotify'). "
    "Keep time expressions in their raw source format (numbers as written) and drop the "
    "'am'/'pm' marker rather than translating it literally, since plain am/pm is an "
    "English borrowing, not standard in {lang_name} (e.g. 'six and nine am' -> 'six et "
    "neuf'). "
    "Do not add, remove, or explain anything. "
    "Reply with ONLY the translated sentence: no quotes, no notes, no alternatives.\n\n"
    "Examples:\n{examples_block}"
)


def build_examples_block(lang_code):
    lines = []
    for en, translated in EXAMPLES.get(lang_code, []):
        lines.append(f"EN: {en}\n{lang_code.upper()}: {translated}")
    return "\n\n".join(lines)


THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
# Heuristics for a hybrid-thinking model leaking its chain-of-thought as plain text
# (no <think> tags) instead of -- or in addition to -- a final answer.
REASONING_MARKERS = re.compile(
    r"^\s*(here'?s a thinking process|let'?s (think|analyze)|step \d|\d+\.\s+\*\*)",
    re.IGNORECASE,
)


def clean_translation(raw_out):
    out = THINK_RE.sub("", raw_out).strip()
    out = out.strip('"').strip("'").strip()
    return out


def translate_one(client, model, text, lang_code, extra_body=None, temperature=TEMPERATURE, max_retries=5):
    lang_name = LANGUAGES[lang_code]
    system = SYSTEM_PROMPT.format(lang_name=lang_name, examples_block=build_examples_block(lang_code))
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": text},
    ]
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                extra_body=extra_body or {},
            )
            out = clean_translation(resp.choices[0].message.content)
            if out and not REASONING_MARKERS.search(out):
                return out
            last_err = RuntimeError(f"looks like leaked reasoning, not a translation: {out[:120]!r}")
        except Exception as e:  # noqa: BLE001
            last_err = e
        time.sleep(min(2 ** attempt, 20))
    raise RuntimeError(f"Translation failed for {text!r}: {last_err}")


### Checkpointed, concurrent translation of a whole split, by one model, into one language

In [7]:
def translate_split(job_name, dataset, lang_code, model_key, position=None):
    client = clients[model_key]
    model = MODELS[model_key]["model"]
    extra_body = MODELS[model_key].get("extra_body", {})

    out_path = OUT_DIR / f"{job_name}_{lang_code}_{model_key}.jsonl"
    done = {}
    if out_path.exists():
        with out_path.open() as f:
            for line in f:
                row = json.loads(line)
                done[row["idx"]] = row

    todo = [i for i in range(len(dataset)) if i not in done]
    print(f"[{job_name}/{lang_code}/{model_key}] {len(done)} cached, {len(todo)} to translate via {model}")

    if todo:
        with out_path.open("a") as f, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {
                ex.submit(translate_one, client, model, dataset[i]["text"], lang_code, extra_body): i
                for i in todo
            }
            desc = f"{model_key}"
            bar = tqdm(as_completed(futures), total=len(futures), desc=desc, position=position, leave=True)
            for fut in bar:
                bar.set_description(f"{model_key}: {job_name}/{lang_code}")
                i = futures[fut]
                text_translated = fut.result()
                row = {
                    "idx": i,
                    "text": dataset[i]["text"],
                    "text_translated": text_translated,
                    "label": dataset[i]["label"],
                    "label_text": dataset[i]["label_text"],
                }
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                f.flush()
                done[i] = row

    return [done[i] for i in range(len(dataset))]


## Smoke test

Translate a handful of examples with both models before committing to a full run.

In [8]:
sample = raw["test"].select(range(5))
for lang_code in LANGUAGES:
    for model_key in MODELS:
        rows = translate_split("smoketest", sample, lang_code, model_key)
        for r in rows:
            print(f"[{lang_code}/{model_key}] {r['text']!r}  ->  {r['text_translated']!r}")
        print()


[smoketest/fr/gemma] 5 cached, 0 to translate via google/gemma-4-31B-it
[fr/gemma] 'tell me time of alarm you set'  ->  "dis-moi l'heure de l'alarme que tu as reglee"
[fr/gemma] 'list all of my alarms'  ->  'liste toutes mes alarmes'
[fr/gemma] 'alarm settings'  ->  "paramètres de l'alarme"
[fr/gemma] 'what alarms are set for today'  ->  "quelles alarmes sont reglees pour aujourd'hui"
[fr/gemma] 'please see see for me the alarms that you have set sunday morning'  ->  "s'il te plaît regarde regarde pour moi les alarmes que tu as reglees dimanche matin"

[smoketest/fr/qwen] 5 cached, 0 to translate via Qwen/Qwen3.6-27B-FP8
[fr/qwen] 'tell me time of alarm you set'  ->  "dis-moi l'heure de l'alarme que tu as reglee"
[fr/qwen] 'list all of my alarms'  ->  'liste toutes mes alarmes'
[fr/qwen] 'alarm settings'  ->  "paramètres d'alarme"
[fr/qwen] 'what alarms are set for today'  ->  "quelles alarmes sont reglees pour aujourd'hui"
[fr/qwen] 'please see see for me the alarms that you have set 

## Full run

Each model hits its own server, so gemma and qwen run in parallel (one thread per
model), each working through `test` (faster feedback) then `train`, both languages.
Within a model, requests to its server are still capped at `MAX_WORKERS` concurrent.
Each model gets a fixed progress-bar row (`position`) so the two bars update in place
side by side instead of clobbering each other's output -- they will very likely finish
at different times since the two servers/models have different throughput.
Safe to re-run / resume — already-translated rows are skipped.


In [9]:
def run_model_jobs(model_key, position):
    results = {}
    for split_name in ["test", "train"]:
        for lang_code in LANGUAGES:
            results[(split_name, lang_code, model_key)] = translate_split(
                split_name, raw[split_name], lang_code, model_key, position=position
            )
    return results


translated = {}
with ThreadPoolExecutor(max_workers=len(MODELS)) as ex:
    futures = {
        ex.submit(run_model_jobs, model_key, position): model_key
        for position, model_key in enumerate(MODELS)
    }
    for fut in as_completed(futures):
        translated.update(fut.result())


[test/fr/gemma] 1076 cached, 0 to translate via google/gemma-4-31B-it
[test/fr/qwen] 1076 cached, 0 to translate via Qwen/Qwen3.6-27B-FP8
[test/es/gemma] 1076 cached, 0 to translate via google/gemma-4-31B-it
[test/es/qwen] 1076 cached, 0 to translate via Qwen/Qwen3.6-27B-FP8
[train/fr/gemma] 8954 cached, 0 to translate via google/gemma-4-31B-it
[train/fr/qwen] 8954 cached, 0 to translate via Qwen/Qwen3.6-27B-FP8
[train/es/gemma] 8954 cached, 0 to translate via google/gemma-4-31B-it
[train/es/qwen] 8954 cached, 0 to translate via Qwen/Qwen3.6-27B-FP8


## Assemble into one dataset per language (4 columns: `text`, `label`, `text_gemma`, `text_qwen`)


In [10]:
model_keys = list(MODELS)  # e.g. ["gemma", "qwen"]

final = {}
for lang_code in LANGUAGES:
    dd = DatasetDict()
    for split_name in ["train", "test"]:
        by_model = {mk: translated[(split_name, lang_code, mk)] for mk in model_keys}
        n = len(by_model[model_keys[0]])
        rows = []
        for i in range(n):
            row = {
                "text": by_model[model_keys[0]][i]["text"],
                "label": by_model[model_keys[0]][i]["label"],
            }
            for mk in model_keys:
                row[f"text_{mk}"] = by_model[mk][i]["text_translated"]
            rows.append(row)
        dd[split_name] = Dataset.from_list(rows)
    final[lang_code] = dd
    print(lang_code, dd)


fr DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'text_gemma', 'text_qwen'],
        num_rows: 8954
    })
    test: Dataset({
        features: ['text', 'label', 'text_gemma', 'text_qwen'],
        num_rows: 1076
    })
})
es DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'text_gemma', 'text_qwen'],
        num_rows: 8954
    })
    test: Dataset({
        features: ['text', 'label', 'text_gemma', 'text_qwen'],
        num_rows: 1076
    })
})


## Spot-check quality

In [11]:
lang_code = "fr"
split_name = "test"
idxs = random.sample(range(len(final[lang_code][split_name])), 10)
for i in idxs:
    row = final[lang_code][split_name][i]
    print("EN:", row["text"])
    for mk in MODELS:
        print(f"{lang_code.upper()} [{mk}]:", row[f"text_{mk}"])
    print("label:", row["label"])
    print()


EN: today's jokes
FR [gemma]: les blagues d'aujourd'hui
FR [qwen]: les blagues d'aujourd'hui
label: 21

EN: get me a taxi to the airport right now
FR [gemma]: commande-moi un taxi pour l'aéroport tout de suite
FR [qwen]: commande-moi un taxi pour l'aéroport tout de suite
label: 60

EN: yeah wonderful response to command.
FR [gemma]: ouais super réponse à la commande.
FR [qwen]: oui, réponse magnifique à la commande.
label: 16

EN: does not matter for me.
FR [gemma]: ça m'est égal.
FR [qwen]: ça m'est égal.
label: 19

EN: what were the cities affected by the earthquake
FR [gemma]: quelles étaient les villes touchées par le tremblement de terre
FR [qwen]: quelles étaient les villes touchées par le séisme
label: 41

EN: remove the board meeting and reschedule for next wednesday
FR [gemma]: supprime la réunion du conseil et reprogramme-la pour mercredi prochain
FR [qwen]: supprime la reunion du conseil et reprogramme pour mercredi prochain
label: 7

EN: have you come across any new recipes

## Save as one HF dataset with a subset per language

Locally, each language becomes its own `save_to_disk` directory (mirrors a subset).
To publish as a single dataset repo with per-language **configs**, push each
`DatasetDict` with `config_name=lang_code` -- left commented out, uncomment and set
your own repo id if you want to publish.


In [13]:
SAVE_DIR = Path("translations/hwu64_final")
for lang_code, dd in final.items():
    dd.save_to_disk(str(SAVE_DIR / lang_code))

repo_id = "DeepPavlov/hwu64-translated"
for lang_code, dd in final.items():
    dd.push_to_hub(repo_id, config_name=lang_code)


Saving the dataset (0/1 shards):   0%|          | 0/8954 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1076 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/8954 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1076 [00:00<?, ? examples/s]

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


...

Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            